## scMAVERICS - snATACseq processing

**🚨 HALT! 🚨**

This section assumes you have completed the previous section, [snRNAseq preprocessing](snRNAseq.md), with the object `06_QC_filtered_anndata_rna.h5ad` in the `atlas` folder. If not, go back and complete the analysis, preferably with the Snakemake pipeline.

This analysis is less standard than the previous page, and subsequently this analysis 

This notebook and all others, can be created from the instructions [here](https://github.com/NIH-CARD/scMAVERICS/tree/main#local-environment)

## Package and data loading

In [1]:
import anndata as ad
import pandas as pd
import scanpy as sc
import snapatac2 as snap
import scipy

import numpy as np
import polars as pl
import pyranges as pr


### Define working and data directories

In [2]:
data_dir = '/data/CARD_singlecell/Brain_atlas/SN_Multiome/'
work_dir = '/data/CARD_singlecell/SN_atlas/' 

### Load metadata

As stated before, metadata should be imported, for visualizing parameters, 

In [3]:
metadata_df = pd.read_csv(work_dir + 'input/metadata.csv')
metadata_df

,Sample_ID,Use_batch,Age,PMI,Sex,Brain_bank,Primary Diagnosis
0,6178,6,68,22.5,Male,Human Brain and Spinal Fluid Resource Center,control
1,3284,1,78,11.5,Female,Human Brain and Spinal Fluid Resource Center,control
2,6384,3,50,22.3,Female,Human Brain and Spinal Fluid Resource Center,PD
3,5638,3,77,19.3,Male,Human Brain and Spinal Fluid Resource Center,PD


In [4]:
# Define keys
sample_key = 'Sample_ID'
batch_key = 'Use_batch'

# Define list of samples
samples = metadata_df[sample_key]
batches = metadata_df[batch_key]
# Ensure sample names are strings
metadata_df[sample_key] = metadata_df[sample_key].astype(str)

In [7]:
step_1_files = [data_dir + f'/batch{batches[i]}/Multiome/{samples[i]}-ARC/outs/02_{samples[i]}_anndata_atac.h5ad' for i in range(len(samples))]
step_1_files

['/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch6/Multiome/6178-ARC/outs/02_6178_anndata_atac.h5ad',
 '/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch1/Multiome/3284-ARC/outs/02_3284_anndata_atac.h5ad',
 '/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch3/Multiome/6384-ARC/outs/02_6384_anndata_atac.h5ad',
 '/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch3/Multiome/5638-ARC/outs/02_5638_anndata_atac.h5ad']

In [9]:
# Read in anndata objects using backed mode
data_dir+'{batch}/Multiome/{sample}/outs/atac_fragments.tsv.gz'
fragments = [data_dir + f'/batch{batches[i]}/Multiome/{samples[i]}-ARC/outs/atac_fragments.tsv.gz' for i in range(len(samples))]
fragments

['/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch6/Multiome/6178-ARC/outs/atac_fragments.tsv.gz',
 '/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch1/Multiome/3284-ARC/outs/atac_fragments.tsv.gz',
 '/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch3/Multiome/6384-ARC/outs/atac_fragments.tsv.gz',
 '/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch3/Multiome/5638-ARC/outs/atac_fragments.tsv.gz']

## atac_merge

In [12]:
adatas = snap.pp.import_fragments(
        fragments, 
        file=step_1_files, 
        chrom_sizes=snap.genome.hg38.chrom_sizes,
        sorted_by_barcode=False,
        min_num_fragments=1000,
        )
adatas

100%|██████████| 4/4 [04:27<00:00, 66.87s/it] 


[AnnData object with n_obs x n_vars = 10502 x 0 backed at '/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch6/Multiome/6178-ARC/outs/02_6178_anndata_atac.h5ad'
     obs: 'n_fragment', 'frac_dup', 'frac_mito'
     uns: 'reference_sequences'
     obsm: 'fragment_paired',
 AnnData object with n_obs x n_vars = 9891 x 0 backed at '/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch1/Multiome/3284-ARC/outs/02_3284_anndata_atac.h5ad'
     obs: 'n_fragment', 'frac_dup', 'frac_mito'
     uns: 'reference_sequences'
     obsm: 'fragment_paired',
 AnnData object with n_obs x n_vars = 8346 x 0 backed at '/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch3/Multiome/6384-ARC/outs/02_6384_anndata_atac.h5ad'
     obs: 'n_fragment', 'frac_dup', 'frac_mito'
     uns: 'reference_sequences'
     obsm: 'fragment_paired',
 AnnData object with n_obs x n_vars = 7710 x 0 backed at '/data/CARD_singlecell/Brain_atlas/SN_Multiome//batch3/Multiome/5638-ARC/outs/02_5638_anndata_atac.h5ad'
     obs: 'n_fragment

In [13]:
# Transcription start site filter
snap.metrics.tsse(adatas, snap.genome.hg38)
# Minimum number of fragments filter
snap.pp.filter_cells(adatas, min_tsse=2.5)


  0%|                                              | 0.00/35.0M [00:00<?, ?B/s]

100%|██████████| 4/4 [00:19<00:00,  4.79s/it]


In [ ]:
adataset.obs_names = np.array(adataset.obs_names) + '_' + np.array(adataset.obs['sample'])
adataset.obs['n_fragment'] = adataset.adatas.obs['n_fragment']
adataset.obs['tsse'] = adataset.adatas.obs['tsse']

## atac_fragment_pseudobulk

In [ ]:
rna = sc.read_h5ad(work_dir + '/atlas/06_QC_filtered_anndata_rna.h5ad')

## atac_celltype_call_peaks

## atac_consensus_peaks

## atac_merged_consensus_peak

## atac_spectral